# Model calibration

*añadir explicación*


### Set up

In [ ]:
# load the required libraries
using DataFrames
using CSV
using Statistics
using JuMP
using Gurobi

project_root = dirname(@__DIR__)

# load the function that models the electricity market and the other auxiliary functions
include(joinpath(project_root, "scripts", "model_electricity_market.jl"))
include(joinpath(project_root, "scripts", "auxiliar_functions.jl"))

# load the fixed datasets
historical_data        = CSV.read(joinpath(project_root, "data", "historical_data.csv"), DataFrame)
technology_data        = CSV.read(joinpath(project_root, "data", "technology_data.csv"), DataFrame)
projection_deltas_data = CSV.read(joinpath(project_root, "data", "projection_deltas_data.csv"), DataFrame)

### Running the model for historical data

Instead of projecting the historical dataset to the future as in the MC loop, we just run it once to 

In [ ]:
# set parameters
baseline_years = [2023, 2024]
scenario_params = scenario_dict["baseline"]

technical_params = (
    elas_residential = 0.015,
    elas_commercial = 0.03,
    elas_industrial = 0.05,
    grid_loss_factor = 0.015,
    coal_min = 0.15,
    coal_max = 0.65,
    coal_ramp = 0.05,
    ccgt_min = 0.05,
    ccgt_max = 1.0,
    ccgt_ramp = 0.25,
    cogen_min = 0.15,
    cogen_max = 0.6,
    cogen_ramp = 0.1,
    diesel_min = 0.2,
    non_ren_waste_max = 0.6,
    non_ren_waste_ramp = 0.01,
    conv_hydro_ramp = 0.1,
    ror_lo_high = 0.3,
    ror_hi_high = 0.5,
    ror_lo_med_high = 0.2,
    ror_hi_med_high = 0.4,
    ror_lo_med_low = 0.15,
    ror_hi_med_low = 0.3,
    ror_lo_low = 0.1,
    ror_hi_low = 0.15,
    ror_ramp = 0.2,
    ph_storage_cap_gwh = 50.0,
    ph_roundtrip_eff = 0.75,
    batt_roundtrip_eff = 0.9,
    batt_decay = 0.001,
    batt_duration = 4.0,
    other_ren_min = 0.25,
    other_ren_max = 0.6,
    other_ren_ramp = 0.05,
    ren_waste_min = 0.0,
    ren_waste_max = 0.65,
    ren_waste_ramp = 0.05
    )

    
# sample time window from historical data (could skip and do full year)
sampled_window_data = sample_time_window(historical_data, baseline_years)

# compute iteration-specific parameters
iteration_params = compute_iteration_params(
        projected  = sampled_window_data,    # sampled window of historical (not projected) data
        technology = technology_data,        # fixed technical and economic parameters by generation technology
        technical  = technical_params,       # model calibration parameters shared across scenarios
        scenario   = scenario_params         # set to the baseline case
        )

# solve the model
results = dispatch_electricity_market(
        projected  = sampled_window_data,    # hourly projected data for 2030
        technology = technology_data,        # fixed technical and economic parameters by generation technology
        technical  = technical_params,       # model calibration parameters shared across scenarios
        scenario   = scenario_params,        # scenario-specific parameters
        iteration  = iteration_params        # iteration-specific parameters
        )

### Compare model results with historical data

Esto tendría que revisarlo todo

In [ ]:
function compute_stats(data, vars; prefix="")
    out = DataFrame(variable = String[])
    
    for var in vars
        values = data isa Dict ? data[var] : data[!, var]
        
        push!(out, (
            variable = var,
            min = minimum(values),
            mean = mean(values),
            median = median(values),
            max = maximum(values),
            std = std(values)
        ))
    end
    
    rename!(out, names(out) .=> (n -> n == "variable" ? n : n * "_" * prefix))
    return out
end

# variables por las que queremos comprobar resultados
variables = [
    "price", "total_demand", "residential_demand", "commercial_demand", "industrial_demand",
    "total_generation",
    "coal_gen", "combined_cycle_gen", "cogeneration_gen", "diesel_gen", "non_renewable_waste_gen",
    "nuclear_gen", "conventional_hydro_gen", "run_of_river_hydro_gen",
    "pumped_hydro_out", "solar_pv_gen", "solar_thermal_gen", "wind_gen",
    "other_renewable_gen", "renewable_waste_gen",
    "battery_out",
    "imports_FRA", "imports_POR", "imports_MOR",
    "share_renewable_gen"
]


model_stats = compute_stats(results, variables; prefix="model")

# Generación total
generation_cols = [
    :coal_gen_gwh, :combined_cycle_gen_gwh, :gas_turbine_gen_gwh, :vapor_turbine_gen_gwh,
    :cogeneration_gen_gwh, :diesel_gen_gwh, :non_renewable_waste_gen_gwh,
    :nuclear_gen_gwh, :conventional_hydro_gen_gwh, :run_of_river_hydro_gen_gwh,
    :pumped_hydro_out_gwh,
    :solar_pv_gen_gwh, :solar_thermal_gen_gwh, :wind_gen_gwh,
    :other_renewable_gen_gwh, :renewable_waste_gen_gwh,
    :battery_out_gwh
]

sampled_window_data.total_generation_gwh =
    [sum(row) for row in eachrow(select(sampled_window_data, generation_cols))]

# Share renovables
renewable_cols = [
    :conventional_hydro_gen_gwh, :run_of_river_hydro_gen_gwh,
    :pumped_hydro_out_gwh,
    :solar_pv_gen_gwh, :solar_thermal_gen_gwh, :wind_gen_gwh,
    :other_renewable_gen_gwh, :renewable_waste_gen_gwh
]

sampled_window_data.share_renewable_gen =
    [total > 0 ? sum(row[renewable_cols]) / total : 0
     for (row, total) in zip(eachrow(sampled_window_data), sampled_window_data.total_generation_gwh)]

var_map = Dict(
    "price" => :spot_price_eur_mwh,  # aquí ojo: sigue en €/MWh
    "total_demand" => :total_demand_gwh,
    "residential_demand" => :residential_demand_gwh,
    "commercial_demand" => :commercial_demand_gwh,
    "industrial_demand" => :industrial_demand_gwh,
    "total_generation" => :total_generation_gwh,
    "coal_gen" => :coal_gen_gwh,
    "combined_cycle_gen" => :combined_cycle_gen_gwh,
    "cogeneration_gen" => :cogeneration_gen_gwh,
    "diesel_gen" => :diesel_gen_gwh,
    "non_renewable_waste_gen" => :non_renewable_waste_gen_gwh,
    "nuclear_gen" => :nuclear_gen_gwh,
    "conventional_hydro_gen" => :conventional_hydro_gen_gwh,
    "run_of_river_hydro_gen" => :run_of_river_hydro_gen_gwh,
    "pumped_hydro_out" => :pumped_hydro_out_gwh,
    "solar_pv_gen" => :solar_pv_gen_gwh,
    "solar_thermal_gen" => :solar_thermal_gen_gwh,
    "wind_gen" => :wind_gen_gwh,
    "other_renewable_gen" => :other_renewable_gen_gwh,
    "renewable_waste_gen" => :renewable_waste_gen_gwh,
    "battery_out" => :battery_out_gwh,
    "imports_FRA" => :imports_France_gwh,
    "imports_POR" => :imports_Portugal_gwh,
    "imports_MOR" => :imports_Morocco_gwh,
    "share_renewable_gen" => :share_renewable_gen
)

real_stats = compute_stats(
    sampled_window_data,
    [var_map[v] for v in variables];
    prefix="real"
)

real_stats.variable = variables

combined_stats = innerjoin(model_stats, real_stats, on="variable")

### Gráficos de comparativa

A partir de aquí también tendría que revisarlo todo

In [ ]:
# Filter the results dictionary to include only the specified keys, to being able to make the graphs
filtered_results = Dict(
    "residential_demand" => results["residential_demand"],
    "commercial_demand" => results["commercial_demand"],
    "industrial_demand" => results["industrial_demand"],
    "price" => results["price"],
    "coal_gen" => results["coal_gen"],
    "combined_cycle_gen" => results["combined_cycle_gen"],
    "gas_turbine_gen" => results["gas_turbine_gen"],
    "vapor_turbine_gen" => results["vapor_turbine_gen"],
    "cogeneration_gen" => results["cogeneration_gen"],
    "diesel_gen" => results["diesel_gen"],
    "non_renewable_waste_gen" => results["non_renewable_waste_gen"],
    "conventional_hydro_gen" => results["conventional_hydro_gen"],
    "run_of_river_hydro_gen" => results["run_of_river_hydro_gen"],
    "pumped_hydro_gen" => results["pumped_hydro_gen"],
    "nuclear_gen" => results["nuclear_gen"],
    "solar_pv_gen" => results["solar_pv_gen"],
    "solar_thermal_gen" => results["solar_thermal_gen"],
    "wind_gen" => results["wind_gen"],
    "other_renewable_gen" => results["other_renewable_gen"],
    "renewable_waste_gen" => results["renewable_waste_gen"],
    "battery_gen" => results["battery_gen"],
    "imports_FRA" => results["imports_FRA"],
    "imports_POR" => results["imports_POR"],
    "imports_MOR" => results["imports_MOR"],
    "exports_FRA" => results["exports_FRA"],
    "exports_POR" => results["exports_POR"],
    "exports_MOR" => results["exports_MOR"],
    "share_renewable_gen" => results["share_renewable_gen"],
    )

# Convert the filtered results dictionary into a DataFrame
results_df = DataFrame(filtered_results)

# Add a time column for better visualization
results_df[!, :time] = 1:size(results_df, 1)




In [ ]:
# Plot actual price vs model-solved price across time
plot(1:length(toy_data.spot_price_eur_mwh), toy_data.spot_price_eur_mwh, 
  label="Actual Price (€/MWh)", xlabel="Time (hours)", ylabel="Price (€/MWh)", legend=:topright)
plot!(results_df[!, :time], results_df[!, :price], label="Model-Solved Price (€/MWh)")

In [ ]:
# Determine the common y-axis range
y_min = min(minimum(toy_data.residential_demand_mwh), minimum(toy_data.industrial_demand_mwh))
y_max = max(maximum(toy_data.residential_demand_mwh), maximum(toy_data.industrial_demand_mwh))

# Plot actual vs model-solved residential demand
plot(1:length(toy_data.residential_demand_mwh), toy_data.residential_demand_mwh, 
    label="Actual residential demand (MWh)", xlabel="Time (hours)", ylabel="Demand (MWh)", legend=:topright, 
    title="Residential Demand", ylim=(y_min, y_max), yformatter = :plain)
plot!(results_df[!, :time], results_df[!, :residential_demand], label="Model-Solved residential demand (MWh)")

In [ ]:
#Plot actual vs model-solved commercial demand
plot(1:length(toy_data.commercial_demand_mwh), toy_data.commercial_demand_mwh, 
    label="Actual commercial demand (MWh)", xlabel="Time (hours)", ylabel="Demand (MWh)", legend=:topright, 
    title="commercial Demand", ylim=(y_min, y_max), yformatter = :plain)
plot!(results_df[!, :time], results_df[!, :commercial_demand], label="Model-Solved commercial demand (MWh)")

In [ ]:
# Plot actual vs model-solved industrial demand
plot(1:length(toy_data.industrial_demand_mwh), toy_data.industrial_demand_mwh, 
    label="Actual industrial demand (MWh)", xlabel="Time (hours)", ylabel="Demand (MWh)", legend=:topright, 
    title="Industrial Demand", ylim=(y_min, y_max), yformatter = :plain)
plot!(results_df[!, :time], results_df[!, :industrial_demand], label="Model-Solved industrial demand (MWh)")

In [ ]:
# Plot actual coal vs model-solved coal across time
plot(1:length(toy_data.coal_gen_mwh), toy_data.coal_gen_mwh, 
label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Coal generation (MWh)", legend=:topright)
plot!(results_df[!, :time], results_df[!, :coal_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual coal vs model-solved coal across time
plot(1:length(toy_data.cogeneration_gen_mwh), toy_data.cogeneration_gen_mwh, 
label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Cogeneration generation (MWh)", legend=:topright)
plot!(results_df[!, :time], results_df[!, :cogeneration_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual combined cycle generation vs model-solved combined cycle generation across time
plot(1:length(toy_data.combined_cycle_gen_mwh), toy_data.combined_cycle_gen_mwh, 
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Natural gas generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :combined_cycle_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual combined cycle generation vs model-solved combined cycle generation across time
plot(1:length(toy_data.nonrenewable_waste_gen_mwh), toy_data.nonrenewable_waste_gen_mwh, 
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Non-renewable waste generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :non_renewable_waste_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual nuclear vs model-solved nuclear across time
plot(1:length(toy_data.nuclear_gen_mwh), toy_data.nuclear_gen_mwh, 
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Nuclear generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :nuclear_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual hydro vs model-solved conventional hydro across time
plot(1:length(toy_data.conventional_hydro_gen_mwh), toy_data.conventional_hydro_gen_mwh,
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Conventional hydro generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :conventional_hydro_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual run of river hydro vs model-solved run of river hydro across time
plot(1:length(toy_data.run_of_river_hydro_gen_mwh), toy_data.run_of_river_hydro_gen_mwh,
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Run of river hydro generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :run_of_river_hydro_gen], label="Model-Solved generation (MWh)")


In [ ]:
# Plot actual pumped hydro vs model-solved pumped hydro across time with transparency for model line
plot(1:length(toy_data.pumped_hydro_gen_mwh), toy_data.pumped_hydro_gen_mwh,
    label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Pumped hydro generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :pumped_hydro_gen], label="Model-Solved generation (MWh)", alpha=0.5)

In [ ]:
# Plot actual solar_pv vs model-solved solar_pv across time
plot(1:length(toy_data.solar_pv_gen_mwh), toy_data.solar_pv_gen_mwh, 
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Solar PV generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :solar_pv_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual solar_thermal vs model-solved solar_thermal across time
plot(1:length(toy_data.solar_thermal_gen_mwh), toy_data.solar_thermal_gen_mwh, title="Solar thermal generation",
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Solar thermal generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :solar_thermal_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual wind vs model-solved wind across time
plot(1:length(toy_data.wind_gen_mwh), toy_data.wind_gen_mwh, title="Wind generation",
  label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Wind generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :wind_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual other renewable vs model-solved other renewable across time
plot(1:length(toy_data.other_renewable_gen_mwh), toy_data.other_renewable_gen_mwh, label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Other renewable generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :other_renewable_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual renewable waste vs model-solved renewable waste across time
plot(1:length(toy_data.renewable_waste_gen_mwh), toy_data.renewable_waste_gen_mwh, label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Renewable waste generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :renewable_waste_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual battery use vs model-solved battery use across time
plot(1:length(toy_data.batteries_gen_mwh), toy_data.batteries_gen_mwh, label="Actual generation (MWh)", xlabel="Time (hours)", ylabel="Batteries generation (MWh)", yformatter = :plain, legend=:topright)
plot!(results_df[!, :time], results_df[!, :battery_gen], label="Model-Solved generation (MWh)")

In [ ]:
# Plot actual imports from France vs model-solved imports from France across time
plot(1:length(toy_data.imports_France_mwh), toy_data.imports_France_mwh, 
  label="Actual imports (MWh)", xlabel="Time (hours)", ylabel="Imports from France (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :imports_FRA], label="Model-Solved imports (MWh)")

In [ ]:
# Plot actual imports from Portugal vs model-solved imports from Portugal across time
plot(1:length(toy_data.imports_Portugal_mwh), toy_data.imports_Portugal_mwh, 
  label="Actual imports (MWh)", xlabel="Time (hours)", ylabel="Imports from Portugal (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :imports_POR], label="Model-Solved imports (MWh)")

In [ ]:
# Plot actual imports from Morocco vs model-solved imports from Morocco across time
plot(1:length(toy_data.imports_Morocco_mwh), toy_data.imports_Morocco_mwh, 
  label="Actual imports (MWh)", xlabel="Time (hours)", ylabel="Imports from Morocco (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :imports_MOR], label="Model-Solved imports (MWh)")

In [ ]:
# Plot actual exports to France vs model-solved exports to France across time
plot(1:length(toy_data.exports_France_mwh), toy_data.exports_France_mwh, 
  label="Actual exports (MWh)", xlabel="Time (hours)", ylabel="Exports to France (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :exports_FRA], label="Model-Solved exports (MWh)")

In [ ]:
# Plot actual exports from Portugal vs model-solved exports from Portugal across time
plot(1:length(toy_data.exports_Portugal_mwh), toy_data.exports_Portugal_mwh, 
  label="Actual exports (MWh)", xlabel="Time (hours)", ylabel="Exports to Portugal (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :exports_POR], label="Model-Solved exports (MWh)")

In [ ]:
# Plot actual exports from Morocco vs model-solved exports from Morocco across time
plot(1:length(toy_data.exports_Morocco_mwh), toy_data.exports_Morocco_mwh, 
  label="Actual exports (MWh)", xlabel="Time (hours)", ylabel="Exports to Morocco (MWh)", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :exports_MOR], label="Model-Solved exports (MWh)")

In [ ]:
# Plot actual share of renewable generation vs model-solved share of renewable generation across time
plot(1:length(toy_data.share_renewable_gen), toy_data.share_renewable_gen, 
  label="Actual share of renewable generation", xlabel="Time (hours)", ylabel="Share of renewable generation", yformatter = :plain, legend=:topright)
plot!(1:length(results_df[!, :time]), results_df[!, :share_renewable_gen], label="Model-Solved share of renewable generation")   

# Add a line which is the mean share of renewable generation in the plot
mean_share = mean(results_df[!, :share_renewable_gen])
plot!(1:length(results_df[!, :time]), fill(mean_share, length(results_df[!, :time])), label="Mean Share", linestyle=:dash, color=:red)

In [ ]:
mean(toy_data.cost_coal_eur_mwh), mean(toy_data.cost_gas_eur_mwh), mean(toy_data.cost_diesel_eur_mwh), mean(toy_data.cost_uranium_eur_mwh)

In [ ]:
# Marginal cost of coal
63 * 0.82 + 8.55 + 16.34 / 0.36

In [ ]:
# Marginal cost of gas
63 * 0.49 + 2.29 + 46.54 / 0.57

In [ ]:
mean(toy_data.coal_gen_mwh), minimum(toy_data.coal_gen_mwh)